In [45]:
import pandas as pd
import numpy as np
quantile_value=0.75
data=pd.read_csv('mosquito_climate_pop_land_use_n.csv',low_memory=False)
start_index_t2m=data.columns.get_loc('t2m_1')
start_index_d2m=data.columns.get_loc('d2m_1')
start_index_tp=data.columns.get_loc('tp_1')
start_index_si10=data.columns.get_loc('si10_1')
t2m_mean=data.iloc[:,start_index_t2m:start_index_t2m+12].quantile(quantile_value,axis=1)
d2m_mean=data.iloc[:,start_index_d2m:start_index_d2m+12].quantile(quantile_value,axis=1)
tp_mean=data.iloc[:,start_index_tp:start_index_tp+12].quantile(quantile_value,axis=1)
si10_mean=data.iloc[:,start_index_si10:start_index_si10+12].quantile(quantile_value,axis=1)


data.insert(1,'t2m_mean',t2m_mean)
data.insert(2,'d2m_mean',d2m_mean)
data.insert(3,'tp_mean',tp_mean)
data.insert(4,'si10_mean',si10_mean)
data["Target_Class"]=1
df=data[data["VECTOR"]=="Aedes albopictus"]
#df_aegypti=df.drop(['Unnamed: 0','VECTOR','OCCURRENCE_ID','SOURCE_TYPE','LOCATION_TYPE','POLYGON_ADMIN','Y','X','COUNTRY','COUNTRY_ID','GAUL_AD0','STATUS','Population_Density','land_use_0','land_use_11','land_use_22','land_use_33','land_use_44','land_use_55','land_use_66','land_use_77'],axis=1)
#df_aegypti=df.drop(['Unnamed: 0','VECTOR','OCCURRENCE_ID','SOURCE_TYPE','LOCATION_TYPE','POLYGON_ADMIN','YEAR','Y','X','COUNTRY','COUNTRY_ID','GAUL_AD0','STATUS'],axis=1)
#columns_list=['t2m_mean', 'd2m_mean', 'tp_mean', 'si10_mean','Y','X','Population_Density', 'land_use_0', 'land_use_11',
#    'land_use_22', 'land_use_33', 'land_use_44', 'land_use_55',
#    'land_use_66', 'land_use_77','Target_Class'] # can/add remove Y,X from here
columns_list=['t2m_mean', 'd2m_mean', 'tp_mean', 'si10_mean','Population_Density', 'land_use_0', 'land_use_11',
        'land_use_22', 'land_use_33', 'land_use_44', 'land_use_55',
        'land_use_66', 'land_use_77','Target_Class'] # can/add remove Y,X from here
df_aegypti=df.loc[:,columns_list]
df_aegypti=df_aegypti.dropna()
#df_aegypti=df_aegypti[df_aegypti['Population_Density']>0]
X=df_aegypti.drop('Target_Class',axis=1)
y=df_aegypti['Target_Class']
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=2)
cols=X_train.columns
from sklearn.preprocessing import StandardScaler
#from sklearn.preprocessing import PowerTransformer
scaler=StandardScaler()
#scaler = PowerTransformer(method='yeo-johnson', standardize=True)  # Default: Yeo-Johnson and standardize=True
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
X_train=pd.DataFrame(X_train,columns=[cols])
X_test=pd.DataFrame(X_test,columns=[cols])

In [57]:
import numpy as np
from scipy.spatial.distance import pdist

def calculate_median_gamma(X, subsample_size=1500):
    """
    Calculates the RBF gamma hyperparameter using the Median Heuristic.
    Gamma = 1 / 2 * median(||x - x'||^2)
    """
    # 1. Subsample if the dataset is very large to save memory/time
    if X.shape[0] > subsample_size:
        idx = np.random.choice(X.shape[0], subsample_size, replace=False)
        X_sample = X.iloc[idx]
    else:
        X_sample = X

    # 2. Compute pairwise squared Euclidean distances
    # dists = ||x - x'||^2
    pairwise_distances = pdist(X_sample, metric='euclidean')

    # 3. Find the median of squared distances
    sigma_median = np.median(pairwise_distances)

    # 4. Calculate gamma
    # For OCSVM in sklearn, the kernel is exp(-(gamma/2) * ||x-x'||^2)
    # The heuristic sets gamma to the inverse of the median squared distance
    gamma_heuristic = 1.0 / (2*(sigma_median ** 2))  

    return gamma_heuristic

# Example Usage:
my_gamma = calculate_median_gamma(X_train)
my_gamma

0.024250064498280627

In [63]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
clf=OneClassSVM()

parameters=[{'nu':[0.03,0.1],'kernel':['rbf'],'gamma':[0.024,0.03]}]
grid_search=GridSearchCV(estimator=clf,param_grid=parameters,scoring='accuracy',cv=5,verbose=0)

grid_search.fit(X_train,y_train)

print('GridSearch CV best score : {:.4f}\n\n'.format(grid_search.best_score_))

print('Parameters that give the best results :','\n\n', (grid_search.best_params_))

print('\n\nEstimator that was chosen by the search :','\n\n', (grid_search.best_estimator_))

print('GridSearch CV score on test set: {0:0.4f}'.format(grid_search.score(X_test, y_test)))

best_model=grid_search.best_estimator_
y_pred=best_model.predict(X_test)

print('Accuracy on test set: {0:0.4f}'.format(accuracy_score(y_test, y_pred)))
# Default (average='binary', pos_label=1)
f1 = f1_score(y_test, y_pred)
print(f"F1-score for binary classification (positive class 1): {f1:.2f}")

GridSearch CV best score : 0.9701


Parameters that give the best results : 

 {'gamma': 0.03, 'kernel': 'rbf', 'nu': 0.03}


Estimator that was chosen by the search : 

 OneClassSVM(gamma=0.03, nu=0.03)
GridSearch CV score on test set: 0.9682
Accuracy on test set: 0.9682
F1-score for binary classification (positive class 1): 0.98
